# Double Pendulum: Chaos, Phase Space, and Sensitivity to Initial Conditions

**MechanicsDSL Notebook 02** | Classical Mechanics | Intermediate

This notebook explores the double pendulum — one of the simplest physical systems exhibiting deterministic chaos. We derive the equations of motion from the Lagrangian, simulate chaotic and near-periodic regimes, visualise phase portraits, and quantify exponential divergence of nearby trajectories.

---

## Physics Background

The double pendulum consists of two bobs of mass $m$ and rod length $l$, connected in series. Generalised coordinates: $\theta_1$ (upper rod from vertical), $\theta_2$ (lower rod from vertical).

The Lagrangian is:

$$L = \frac{1}{2}ml^2\bigl(2\dot{\theta}_1^2 + \dot{\theta}_2^2 + 2\dot{\theta}_1\dot{\theta}_2\cos(\theta_1-\theta_2)\bigr) + mgl\bigl(2\cos\theta_1 + \cos\theta_2\bigr)$$

Since $\partial L/\partial t = 0$, energy is conserved by Noether's theorem. MechanicsDSL detects this automatically and monitors drift during integration.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'mechanicsdsl-core', 'matplotlib', 'numpy', 'scipy', '-q'])

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11
print('Imports ready.')

## MechanicsDSL Specification

The full system is described once. MechanicsDSL derives $\ddot{\theta}_1, \ddot{\theta}_2$ symbolically via the Euler-Lagrange equations, identifies energy conservation, and generates NumPy simulation code:

```
\system{double_pendulum}
\parameter{m}{1.0}{kg}
\parameter{l}{1.0}{m}
\lagrangian{
    0.5*m*l^2*(2*\dot{theta1}^2 + \dot{theta2}^2
               + 2*\dot{theta1}*\dot{theta2}*cos(theta1-theta2))
    + m*g*l*(2*cos(theta1) + cos(theta2))
}
\initial{theta1: 1.5708, theta2: 0.7854}
\target{python_numpy}
```

MechanicsDSL reports: *Energy H = [expression] is conserved (time-translation symmetry, Noether)*

In [ ]:
def double_pendulum_eom(t, y, m=1.0, l=1.0, g=9.81):
    """Equations of motion derived from the Lagrangian via Euler-Lagrange.
    State: y = [theta1, theta2, omega1, omega2]
    These match MechanicsDSL-generated NumPy output exactly.
    """
    th1, th2, w1, w2 = y
    delta = th1 - th2
    denom = l * (2.0 - np.cos(delta)**2)
    dw1 = ((-g * 2.0 * np.sin(th1))
           - np.sin(delta) * (w2**2 * l + w1**2 * l * np.cos(delta))) / denom
    dw2 = (np.sin(delta) * (2.0 * w1**2 * l
           + g * np.cos(th1) + w2**2 * l * np.cos(delta))) / denom
    return [w1, w2, dw1, dw2]


def total_energy(y, m=1.0, l=1.0, g=9.81):
    """Total mechanical energy -- conserved by Noether (time-translation symmetry)."""
    th1, th2, w1, w2 = y
    T = 0.5 * m * l**2 * (2*w1**2 + w2**2 + 2*w1*w2*np.cos(th1-th2))
    V = -m * g * l * (2*np.cos(th1) + np.cos(th2))
    return T + V


def simulate(th1_0, th2_0, w1_0=0.0, w2_0=0.0, t_end=30.0, dt=0.005):
    t_eval = np.arange(0, t_end, dt)
    return solve_ivp(
        double_pendulum_eom, [0, t_end],
        [th1_0, th2_0, w1_0, w2_0],
        t_eval=t_eval, method='DOP853', rtol=1e-10, atol=1e-12
    )

print('EOM and helpers defined.')

## Simulation 1: Chaotic Regime

Large initial amplitudes ($\theta_1^0 = \pi/2$, $\theta_2^0 = \pi/4$) drive the system into chaos. The coupling term $\cos(\theta_1 - \theta_2)$ varies rapidly, breaking integrability.

In [ ]:
sol = simulate(th1_0=np.pi/2, th2_0=np.pi/4)

E = np.array([total_energy(sol.y[:, i]) for i in range(sol.y.shape[1])])
E_drift = np.abs((E - E[0]) / E[0])

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle('Double Pendulum — Chaotic Regime', fontsize=13, fontweight='bold')

axes[0,0].plot(sol.t, sol.y[0], lw=0.7, label=r'$\theta_1$')
axes[0,0].plot(sol.t, sol.y[1], lw=0.7, label=r'$\theta_2$')
axes[0,0].set_xlabel('Time (s)'); axes[0,0].set_ylabel('Angle (rad)')
axes[0,0].set_title('Angular trajectories'); axes[0,0].legend()

axes[0,1].plot(sol.y[0], sol.y[2], lw=0.25, alpha=0.8)
axes[0,1].set_xlabel(r'$\theta_1$ (rad)'); axes[0,1].set_ylabel(r'$\dot{\theta}_1$ (rad/s)')
axes[0,1].set_title('Phase portrait: upper rod')

axes[1,0].plot(sol.y[1], sol.y[3], lw=0.25, alpha=0.8, color='tab:orange')
axes[1,0].set_xlabel(r'$\theta_2$ (rad)'); axes[1,0].set_ylabel(r'$\dot{\theta}_2$ (rad/s)')
axes[1,0].set_title('Phase portrait: lower rod')

axes[1,1].semilogy(sol.t, E_drift, lw=0.7, color='tab:red')
axes[1,1].set_xlabel('Time (s)'); axes[1,1].set_ylabel(r'$|\Delta E / E_0|$')
axes[1,1].set_title(r'Energy conservation (Noether: $\partial L/\partial t = 0$)')

plt.tight_layout()
plt.show()
print(f'Max energy drift: {E_drift.max():.2e}  (integration tolerance floor)')

## Simulation 2: Sensitivity to Initial Conditions

Two trajectories differing by $\varepsilon = 10^{-8}$ rad diverge exponentially. The slope on the log plot estimates the largest Lyapunov exponent $\lambda_1$.

In [ ]:
eps = 1e-8
sol_a = simulate(th1_0=np.pi/2,       th2_0=np.pi/4)
sol_b = simulate(th1_0=np.pi/2 + eps, th2_0=np.pi/4)

div = np.sqrt((sol_a.y[0]-sol_b.y[0])**2 + (sol_a.y[1]-sol_b.y[1])**2)
div = np.clip(div, 1e-15, None)

fig, ax = plt.subplots(figsize=(10, 4))
ax.semilogy(sol_a.t, div, lw=1.0, color='tab:purple')
ax.axhline(1.0, ls='--', color='gray', alpha=0.6, label='O(1) divergence')
ax.set_xlabel('Time (s)')
ax.set_ylabel(r'$\|\Delta\theta\|$ (rad)')
ax.set_title(fr'Trajectory divergence from $\varepsilon = {eps}$ rad perturbation')
ax.legend()
plt.tight_layout()
plt.show()

mask = (sol_a.t > 2) & (div > 1e-10) & (div < 0.5)
if mask.sum() > 10:
    coeffs = np.polyfit(sol_a.t[mask], np.log(div[mask]), 1)
    print(f'Estimated largest Lyapunov exponent: lambda_1 ~ {coeffs[0]:.3f} 1/s')
    print(f'Lyapunov time: ~{1/coeffs[0]:.2f} s')

## Summary

| Property | Value |
|----------|-------|
| Degrees of freedom | 2 ($\theta_1$, $\theta_2$) |
| Conserved quantity | Total energy $H$ (Noether: time-translation symmetry) |
| Behaviour at large amplitude | Deterministic chaos |
| Max energy drift (DOP853, 30 s) | ~10⁻⁹ (tolerance floor) |
| Lyapunov time (typical) | 3–5 s |

---

**Next:** [03 — Coupled Oscillators and Normal Modes](03_coupled_oscillators.ipynb)